<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-03/NB03_artificial_moral_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 03: Implementing Artificial Moral Agents

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-03/lab.html>.

## Overview

Theories become engineering once they are implemented. This week surveys top-down, bottom-up and hybrid artificial moral agents, from ethical governors and formally verified agents to systems that learn moral judgments from cases or crowds, and asks what each design can guarantee [1, 2].

## Learning outcomes

By the end of the week, students are expected to describe the architecture of an ethical governor, to explain how case-based and crowd-trained systems learn moral judgments, to identify the failure modes of each approach, and to design a hybrid agent that combines learned proposals, explicit constraints and human oversight.

## Top-down architectures: governors and verification

Arkin designed an ethical governor for autonomous robots that could use lethal force [3]. The governor sits between the planner and the actuators. It evaluates each proposed action against constraints derived from the laws of war and the rules of engagement, and it suppresses actions that violate them. The design makes the constraints inspectable and allows responsibility to be traced to the people who wrote them. Its limits are equally clear: The constraints are only as good as the perception that classifies a person as a civilian or a combatant, and they cover only the situations their authors anticipated.

Formal methods add guarantees to such designs. Dennis and colleagues verified with model checking that an autonomous agent selects the most ethical option available under a prioritised set of ethical principles, and that it acts unethically only when no ethical option exists [4]. Verification proves properties of the decision logic. It cannot prove that the logic's inputs describe the world correctly, which is why perception and specification remain open problems in later weeks.

![Figure 3.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-03/figures/w03_fig1.png)

*Figure 3.1. A hybrid artificial moral agent. A learned policy proposes actions, an ethical governor removes those that violate constraints, and flagged cases are escalated to a human [3, 5].*

## Learning moral judgments from cases and crowds

Bottom-up systems learn from examples. MedEthEx, an early medical ethics advisor, represented cases with duties drawn from Ross's prima facie duties, such as non-maleficence, beneficence and respect for autonomy, and learned a decision principle from cases on which ethicists agreed [6, 7]. The learned principle could then justify its recommendation by citing the duties involved.

Large neural models extended this idea to broad coverage. Delphi was trained on a large collection of crowdsourced moral judgments and produced plausible answers to many everyday questions [8]. Its critics and its authors documented inconsistencies under rephrasing and the reproduction of social biases present in the training judgments. The ETHICS benchmark of Hendrycks and colleagues tests whether language models predict human judgments about justice, well-being, duties, virtues and commonsense morality, and it found promising but incomplete ability [9]. A central lesson is that a learned model reproduces the regularities of its data, including morally irrelevant ones. The notebook makes this concrete with a classifier that absorbs a sampling bias from its annotators.

## What the survey literature shows

Tolmeijer and colleagues surveyed implementations in machine ethics and classified them by the ethical theory they encode, by their implementation approach and by their evaluation [1]. They found that systematic evaluation was rare and that the choice of theory often followed the convenience of the technique rather than philosophical argument. Wallach and Allen had anticipated that practical systems would need both rules and learning [2].

The literature also stresses oversight. Kose proposed a human-supervised model for safe artificial intelligence systems, in which human supervision is part of the control architecture rather than an afterthought [10]. Kose, Cankaya and Yigit listed remarkable issues for ethics and safety in the future of AI, including the difficulty of anticipating the behaviour of learning systems [11].

## Hybrid designs

A hybrid agent can combine the strengths of both families. A learned component proposes candidate actions and ranks them by expected benefit. An explicit governor removes candidates that violate hard constraints and flags uncertain ones. Flagged cases go to a human, and every decision is logged for later audit. The design follows the recommendation of Allen, Smit and Wallach to use learning where rules lack coverage and rules where learning lacks guarantees [5].

Two cautions apply. First, a governor can only filter what it can recognise, so a constraint on deception is useless if the system cannot detect deception. Second, human oversight is effective only if humans have the time, information and authority to intervene. These conditions reappear in Week 14, where the EU AI Act requires human oversight for high-risk systems.

> **Pause and reflect.** In the hybrid design of Figure 3.1, which component would be blamed after a harmful decision, and which component should be?

# Hands-on lab

The notebook trains a moral judgment classifier on synthetic cases whose annotations carry a sampling bias, measures how often its verdicts change when only a morally irrelevant attribute changes, and then adds an explicit governor on top of the learned model [3, 8].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Synthetic cases with a hidden sampling bias

Each case describes an action by expected harm, expected benefit, consent and deception. A reference judgment follows a Ross-style weighing of duties. The attribute `group` describes who made the request and is morally irrelevant, but annotators mostly saw group 1 in acceptable cases.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(SEED)

def make_cases(n, biased):
    harm, benefit = rng.uniform(0, 1, n), rng.uniform(0, 1, n)
    consent, deception, group = rng.integers(0, 2, n), rng.integers(0, 2, n), rng.integers(0, 2, n)
    score = 1.2 * benefit - 1.5 * harm + 0.8 * consent - 1.0 * deception
    y = (score + rng.normal(0, 0.25, n) > 0).astype(int)
    X = np.column_stack([harm, benefit, consent, deception, group])
    if biased:  # group 1 cases were mostly collected when the action was acceptable
        keep = (group == 0) | (y == 1) | (rng.uniform(size=n) < 0.25)
        X, y = X[keep], y[keep]
    return X, y

FEATURES = ["harm", "benefit", "consent", "deception", "group"]
X_train, y_train = make_cases(3000, biased=True)
X_test, y_test = make_cases(2000, biased=False)
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", round(model.score(X_test, y_test), 3))
for f, c in zip(FEATURES, model.coef_[0]):
    print(f"{f:10s} {c:+.2f}")

The coefficient on `group` should be close to zero for a fair judge. The biased collection process made it positive: The model has learned that requests from group 1 are more acceptable.

### Your turn, exercise 1

Compute the counterfactual flip rate: the fraction of test cases whose predicted label changes when only `group` is flipped (0 becomes 1 and 1 becomes 0).

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _flip_rate_reference():
    X_cf = X_test.copy(); X_cf[:, 4] = 1 - X_cf[:, 4]
    return float(np.mean(model.predict(X_test) != model.predict(X_cf)))

In [ ]:
flip_rate = None  # your computation
check("Exercise 1", flip_rate, _flip_rate_reference(), tol=1e-9)

## 2. Adding an explicit governor

A hard constraint says that an action involving deception without consent may never be approved, whatever the learned model predicts.

In [ ]:
pred = model.predict(X_test)
violates = (X_test[:, 3] == 1) & (X_test[:, 2] == 0)
print("approved by the model but forbidden by the constraint:", int(np.sum(pred[violates] == 1)))
governed = np.where(violates, 0, pred)
print("accuracy without governor:", round(np.mean(pred == y_test), 3), " with governor:", round(np.mean(governed == y_test), 3))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(FEATURES, model.coef_[0], color=["#1D5B78"] * 4 + ["#C0801A"])
ax.axhline(0, color="k", lw=0.8); ax.set_title("Learned weights: the last bar is morally irrelevant")
plt.show()

### Your turn, exercise 2

Retrain the model without the `group` column and report its test accuracy and flip rate. Explain in two sentences why removing a protected attribute does not always remove its influence when other features act as proxies.

In [ ]:
# Your code here

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-03/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which constraint in the lab was most costly in utility, and would you keep it? Justify your answer with one of the theories from Week 2.
2. Describe a morally irrelevant attribute that could leak into a model in your own field, and the data collection process that could cause the leak.
3. Where in your own work would a human reviewer add value, and what information would the reviewer need?

## Weekly task and submission

Design a hybrid artificial moral agent for a decision task in your field. Submit a one-page architecture description with a diagram, listing the learned component, at least three explicit constraints, the escalation rule for human oversight and what is logged. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Artificial moral agents after a decade.** Review implemented architectures for artificial moral agents published since 2015 and classify them as top-down, bottom-up or hybrid [1]. Include the debate around Delphi and assess what the systems can and cannot do [8].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Tolmeijer, S., Kneer, M., Sarasua, C., Christen, M., & Bernstein, A. (2021). Implementations in machine ethics: A survey. *ACM Computing Surveys*, 53(6), Article 132. <https://doi.org/10.1145/3419633>

[2] Wallach, W., & Allen, C. (2009). *Moral Machines: Teaching Robots Right from Wrong*. Oxford University Press.

[3] Arkin, R. C. (2009). *Governing Lethal Behavior in Autonomous Robots*. Chapman and Hall/CRC.

[4] Dennis, L., Fisher, M., Slavkovik, M., & Webster, M. (2016). Formal verification of ethical choices in autonomous systems. *Robotics and Autonomous Systems*, 77, 1-14. <https://doi.org/10.1016/j.robot.2015.11.012>

[5] Allen, C., Smit, I., & Wallach, W. (2005). Artificial morality: Top-down, bottom-up, and hybrid approaches. *Ethics and Information Technology*, 7(3), 149-155. <https://doi.org/10.1007/s10676-006-0004-4>

[6] Anderson, M., Anderson, S. L., & Armen, C. (2006). MedEthEx: A prototype medical ethics advisor. In *Proceedings of the 18th Conference on Innovative Applications of Artificial Intelligence (IAAI-06)*. AAAI Press.

[7] Ross, W. D. (1930). *The Right and the Good*. Clarendon Press.

[8] Jiang, L., Hwang, J. D., Bhagavatula, C., et al. (2021). Can machines learn morality? The Delphi experiment. arXiv preprint arXiv:2110.07574. <https://arxiv.org/abs/2110.07574>

[9] Hendrycks, D., Burns, C., Basart, S., Critch, A., Li, J., Song, D., & Steinhardt, J. (2021). Aligning AI with shared human values. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/2008.02275>

[10] Kose, U. (2018). Güvenli yapay zekâ sistemleri için insan denetimli bir model geliştirilmesi [Developing a human-supervised model for safe artificial intelligence systems]. *Mühendislik Bilimleri ve Tasarım Dergisi*, 6(1), 93-107. In Turkish.

[11] Kose, U., Cankaya, I. A., & Yigit, T. (2017). Ethics and safety in the future of artificial intelligence: Remarkable issues. *International Journal of Engineering Science and Application*, 2(2), 65-70.